In [371]:
import serial
import time
import numpy as np
import re

import socket
import os
import binascii
import struct

import matplotlib.pyplot as plt

import serial.tools.list_ports

In [372]:
def udp_config(
    CMD, stage1_addr, underwater_addr, data=None,
    udp_ip='192.168.31.16', udp_port=4660, timeout=5.0,
    verbose=False
):
    if CMD == 'W':
        if data is None:
            raise ValueError("CMD='W' requires data")
        if isinstance(data, int):
            data = bytes([data])
        elif isinstance(data, (list, tuple)):
            data = bytes(data)
        cmd_byte = 0x80
        payload = data
    elif CMD == 'R':
        cmd_byte = 0xC0
        payload = b''
    else:
        raise ValueError("CMD must be 'W' or 'R'")

    packet = bytes([
        0xFF, cmd_byte, 0x00, 0x01,
        0x00, 0x00, underwater_addr, stage1_addr
    ]) + payload

    sock = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
    sock.settimeout(timeout)

    try:
        sock.bind(('', udp_port))
        sock.sendto(packet, (udp_ip, udp_port))

        try:
            raw, addr = sock.recvfrom(1024)
        except socket.timeout:
            raise RuntimeError(
                f"UDP response timeout: target={udp_ip}:{udp_port}"
            )

        if verbose:
            print(f"RX from {addr}, len={len(raw)}, raw={raw.hex(' ')}")

        if len(raw) < 8:
            raise RuntimeError(
                f"Invalid RBCP response length: {len(raw)}, raw={raw.hex(' ')}"
            )

        length = int.from_bytes(raw[2:4], 'big')
        if len(raw) < 8 + length:
            raise RuntimeError(
                f"Invalid RBCP payload length: {len(raw)}, expected >= {8 + length}, "
                f"raw={raw.hex(' ')}"
            )

        header = raw[0]
        cmd_ack = raw[1]

        if header != 0xFF:
            raise RuntimeError(f"Invalid RBCP header: {raw.hex(' ')}")

        is_ack = (cmd_ack >> 3) & 1
        bus_err = cmd_ack & 1

        if not is_ack:
            raise RuntimeError(f"RBCP ACK bit not set: {raw.hex(' ')}")

        if bus_err:
            raise RuntimeError(f"RBCP bus error: {raw.hex(' ')}")

        return raw, raw[8:8 + length]

    finally:
        sock.close()


def udp_write_byte(value, stage1_addr, underwater_addr,
                   udp_ip='192.168.31.16', udp_port=4660, timeout=5.0):
    """Write one slow-control byte and wait for its RBCP ACK."""
    return udp_config(
        'W', stage1_addr, underwater_addr, value,
        udp_ip=udp_ip, udp_port=udp_port, timeout=timeout
    )


def udp_read_word(stage1_addr, underwater_addr,
                  udp_ip='192.168.31.16', udp_port=4660, timeout=5.0):
    """
    Collect one 16-bit slow-control result.

    stage2's asymmetric 16-to-8-bit FIFO emits the high byte first.
    Exactly two RBCP reads are required to keep the FIFO aligned.
    Before reading, poll RBCP address high byte 0xFA until a slow-control
    result is available. This avoids a timeout returning a fake 0x00 and
    corrupting the high/low byte pairing.
    """
    status_addr = 0xFA
    deadline = time.monotonic() + timeout

    while True:
        _raw_status, status_payload = udp_config(
            'R', stage1_addr, status_addr,
            udp_ip=udp_ip, udp_port=udp_port, timeout=timeout
        )
        if len(status_payload) != 1:
            raise RuntimeError(
                f"Unexpected slow-control status length: {len(status_payload)}"
            )
        if status_payload[0] & 0x01:
            break
        if time.monotonic() >= deadline:
            raise RuntimeError(
                f"Slow-control result timeout: stage1={stage1_addr}, "
                f"underwater={underwater_addr}"
            )
        time.sleep(0.005)

    raw_hi, payload_hi = udp_config(
        'R', stage1_addr, underwater_addr,
        udp_ip=udp_ip, udp_port=udp_port, timeout=timeout
    )
    raw_lo, payload_lo = udp_config(
        'R', stage1_addr, underwater_addr,
        udp_ip=udp_ip, udp_port=udp_port, timeout=timeout
    )

    if len(payload_hi) != 1 or len(payload_lo) != 1:
        raise RuntimeError(
            "Unexpected slow-control read length: "
            f"hi={len(payload_hi)}, lo={len(payload_lo)}"
        )

    return (payload_hi[0] << 8) | payload_lo[0]

DAQ_RBCP_API_VERSION = 3

In [379]:
udp_config('W', 1, 1, 0XF0)

(b'\xff\x88\x00\x01\x00\x00\x01\x01\xf0', b'\xf0')

In [373]:
PTP_RESULT_BITS = 384
PTP_RESULT_BYTES = PTP_RESULT_BITS // 8
PTP_API_VERSION = 4


def ptp_start(stage1_addr=1, channel=1,
              udp_ip='192.168.31.16', udp_port=4660,
              timeout=5.0):
    """Send PTP start and wait for the stage2 RBCP ACK."""
    if not 1 <= stage1_addr <= 8:
        raise ValueError("stage1_addr must be in 1..8")
    if not 1 <= channel <= 8:
        raise ValueError("channel must be in 1..8")

    ptp_control_addr = 0xF0 | channel
    raw_ack, payload = udp_config(
        'W', stage1_addr, ptp_control_addr, 0x00,
        udp_ip=udp_ip, udp_port=udp_port, timeout=timeout
    )
    print(
        f"PTP start ACK: stage1={stage1_addr}, "
        f"channel={channel}, cmd_addr=0x{ptp_control_addr:02X}"
    )
    return raw_ack, payload


def ptp_read_data(stage1_addr=1, channel=1,
                  max_bytes=PTP_RESULT_BYTES,
                  udp_ip='192.168.31.16', udp_port=4660,
                  timeout=5.0, idle_timeout=1.0,
                  poll_interval=0.01):
    """
    Read all currently available PTP bytes through non-blocking UDP status
    queries, followed by one-byte RBCP reads.

    RBCP address high byte:
      F1..F8 : pop one byte from fifo_ptp
      F9     : query whether fifo_ptp has at least one byte

    Reading stops after max_bytes, or after fifo_ptp remains empty for
    idle_timeout seconds. This avoids leaving stage2 stuck in RD_WAIT.
    """
    if not 1 <= stage1_addr <= 8:
        raise ValueError("stage1_addr must be in 1..8")
    if not 1 <= channel <= 8:
        raise ValueError("channel must be in 1..8")
    if max_bytes <= 0:
        raise ValueError("max_bytes must be positive")

    ptp_control_addr = 0xF0 | channel
    ptp_status_addr = 0xF9
    data = bytearray()
    idle_since = None

    while len(data) < max_bytes:
        _raw_status, status_payload = udp_config(
            'R', stage1_addr, ptp_status_addr,
            udp_ip=udp_ip, udp_port=udp_port, timeout=timeout
        )
        if len(status_payload) != 1:
            raise RuntimeError(
                f"Unexpected PTP status payload length: {len(status_payload)}"
            )

        if status_payload[0] & 0x01:
            idle_since = None
            _raw_data, payload = udp_config(
                'R', stage1_addr, ptp_control_addr,
                udp_ip=udp_ip, udp_port=udp_port, timeout=timeout
            )
            if len(payload) != 1:
                raise RuntimeError(
                    f"Unexpected PTP data payload length: {len(payload)}"
                )
            data.extend(payload)
        else:
            now = time.monotonic()
            if idle_since is None:
                idle_since = now
            elif now - idle_since >= idle_timeout:
                break
            time.sleep(poll_interval)

    result = bytes(data)
    print(
        f"PTP data: stage1={stage1_addr}, channel={channel}, "
        f"{len(result)} bytes = {len(result) * 8} bits"
    )
    return result


def ptp_start_and_read(stage1_addr=1, channel=1,
                       max_bytes=PTP_RESULT_BYTES,
                       udp_ip='192.168.31.16', udp_port=4660,
                       timeout=5.0, idle_timeout=1.0):
    """Run one complete PTP transaction over UDP."""
    ptp_start(
        stage1_addr=stage1_addr, channel=channel,
        udp_ip=udp_ip, udp_port=udp_port, timeout=timeout
    )
    data = ptp_read_data(
        stage1_addr=stage1_addr, channel=channel,
        max_bytes=max_bytes,
        udp_ip=udp_ip, udp_port=udp_port,
        timeout=timeout, idle_timeout=idle_timeout
    )
    return data, len(data) * 8


In [406]:
if globals().get('PTP_API_VERSION') != 4:
    raise RuntimeError(
        'Old PTP functions are loaded. Restart Kernel and run the PTP '
        'definition cell again.'
    )

ptp_bytes, ptp_bits = ptp_start_and_read(
    stage1_addr=1,
    channel=1
)
print(ptp_bytes.hex(' '))

result = ptp_bytes

t = []
i = 0
arr_timestamp = np.frombuffer(result[48*i:48*i+32], dtype='>u8')#.astype(int)
arr_tdc = np.frombuffer(result[48*i+32:48*i+48], dtype='>u4')#.astype(int)
t.append(arr_timestamp.tolist() + (arr_tdc >> 8).tolist() + (arr_tdc&0xff).tolist())
t = np.array(t).squeeze()
print('ptp timestamps [s]:', t[:4]*8e-9)
print('ptp coarse time [ms]:', t[4:8]*2.5e-6)

dt = ((t[5] - t[4]) - (t[7] - t[6]))/2 *2.5
print('coarse time delay:', dt, 'ns')

PTP start ACK: stage1=1, channel=1, cmd_addr=0xF1
PTP data: stage1=1, channel=1, 48 bytes = 384 bits
00 00 00 79 2c 62 5b 17 00 00 00 77 14 58 e7 19 00 00 00 77 14 58 e7 1b 00 00 00 79 2c 62 5b 47 a6 2c 1f 75 a6 2c ce 61 3d 17 ca 27 3d 17 e8 25
ptp timestamps [s]: [4163.48548934 4091.53983098 4091.539831   4163.48548972]
ptp coarse time [ms]: [27.2256775 27.226115  10.009465  10.00954  ]
coarse time delay: 181.25 ns


In [ ]:
# result = ptp_bytes

# t = []
# i = 0
# arr_timestamp = np.frombuffer(result[48*i:48*i+32], dtype='>u8')#.astype(int)
# arr_tdc = np.frombuffer(result[48*i+32:48*i+48], dtype='>u4')#.astype(int)
# t.append(arr_timestamp.tolist() + (arr_tdc >> 8).tolist() + (arr_tdc&0xff).tolist())
# t = np.array(t).squeeze()
# print('ptp timestamps [s]:', t[:4]*8e-9)
# print('ptp coarse time [ms]:', t[4:8]*2.5e-6)

# dt = ((t[5] - t[4]) - (t[7] - t[6]))/2 *2.5
# print('coarse time delay:', dt, 'ns')

ptp timestamps [s]: [1511.39575449 1503.62394318 1503.62394321 1511.39575486]
ptp coarse time [ms]: [22.9638325 22.964265  25.519195  25.5192675]
coarse time delay: 180.0 ns


In [ ]:
arr_tdc.astype(np.uint64)

array([2351496553, 2351540935, 2613165653, 2613173169], dtype=uint64)

- Si5345

In [375]:
def parse_si5345_txt_file(file_path):
    """
    Parse Si5345 register config text file.

    Returns
    -------
    config_data : bytearray
        Byte array of SPI data, 4 bytes per register:
        [0x00, addr, 0x40, data] with page-switch commands inserted.
    """
    result = bytearray()
    current_page = None
    spi_3wire_value = 0x08

    with open(file_path, 'r') as f:
        lines = f.readlines()

    for line in lines:
        line = line.strip()
        if not line or line.startswith('#') or ',' not in line:
            continue
        try:
            addr_data = line.split(',')
            address = int(addr_data[0].strip(), 16)
            data = int(addr_data[1].strip(), 16)
        except (ValueError, IndexError):
            continue

        if address == 0x002B:
            spi_3wire_value = data | 0x08

        page = (address >> 8) & 0xFF
        addr = address & 0xFF

        if page != current_page:
            result.extend([0x00, 0x01, 0x40, page])
            current_page = page

        result.extend([0x00, addr, 0x40, data])

    result.extend([0x00, 0x01, 0x40, 0x00])
    result.extend([0x00, 0x2B, 0x40, spi_3wire_value])
    return result

def config_si5345(config_data, stage1_addr, underwater_addr,
                   udp_ip='192.168.31.16', udp_port=4660):
    """
    Configure one Si5345 on a given stage1+underwater board.

    Sends the 0xF0 bulk-write command followed by length and config
    bytes one at a time over UDP slow control.

    Parameters
    ----------
    config_data : bytearray
        Output of parse_si5345_txt_file.
    stage1_addr : int
        Stage1 board address, 1~8.
    underwater_addr : int
        Underwater board address, 1~8.
    udp_ip : str
        SiTCP target IP.
    udp_port : int
        SiTCP target UDP port.
    """
    length = len(config_data)

    packet_count = 0
    prev_ack_ok = True  # 上一个包是否已收到ACK
    def _send_byte(b):
        nonlocal packet_count, prev_ack_ok
        assert prev_ack_ok, "上一个包未收到ACK，不能发送下一个包"
        packet_count += 1
        udp_write_byte(
            b, stage1_addr, underwater_addr,
            udp_ip=udp_ip, udp_port=udp_port, timeout=5.0
        )
        prev_ack_ok = True  # udp_config 阻塞等待ACK，返回即表示成功

    # 0xF0 command byte
    _send_byte(0xF0)

    # 2-byte big-endian length
    _send_byte((length >> 8) & 0xFF)
    _send_byte(length & 0xFF)

    # config data bytes, with 300 ms pause after first 20 bytes
    for i, b in enumerate(config_data):
        _send_byte(b)
        if i == 19:
            time.sleep(0.3)

    # RBCP ACK only confirms receipt. Wait for the final SPI word to finish.
    time.sleep(0.05)

    print(f"Si5345 config sent: {length} bytes to stage1={stage1_addr},"
          f" underwater={underwater_addr}")
    return length

def read_si5345(page, addr, stage1_addr, underwater_addr,
                udp_ip='192.168.31.16', udp_port=4660):
    """
    Read one Si5345 register via the 0xF1 slow-control command.

    Sends the 4-word read sequence, then issues exactly 2 collect reads
    (0xC0) to retrieve one 16-bit result from stage2 fifo_sc_tx.

    Parameters
    ----------
    page : str or int
        Si5345 page number, e.g. 0 or '00'.
    addr : str or int
        Si5345 register address within the page (low 8 bits),
        e.g. 0x2B or '2B'.
    stage1_addr : int
        Stage1 board address, 1~8.
    underwater_addr : int
        Underwater board address, 1~8.
    udp_ip : str
        SiTCP target IP.
    udp_port : int
        SiTCP target UDP port.

    Returns
    -------
    result : str
        Hex string of the register value (2 chars, e.g. '08').
    """
    page_val = int(page, 16) if isinstance(page, str) else page
    addr_val = int(addr, 16) if isinstance(addr, str) else addr

    def _send_byte(b):
        udp_write_byte(
            b, stage1_addr, underwater_addr,
            udp_ip=udp_ip, udp_port=udp_port
        )

    # 0xF1 command
    _send_byte(0xF1)

    # 4 × 16-bit SPI words (8 bytes total)
    _send_byte(0x00);  _send_byte(0x01)   # word1: page register addr
    _send_byte(0x40);  _send_byte(page_val) # word2: write page
    _send_byte(0x00);  _send_byte(addr_val) # word3: register addr
    _send_byte(0x80);  _send_byte(0x00)     # word4: read command

    # The asymmetric 16-to-8 FIFO emits the high byte first.
    value = udp_read_word(
        stage1_addr, underwater_addr,
        udp_ip=udp_ip, udp_port=udp_port
    )

    result = f"{value & 0xFF:02X}"
    print(f"Si5345 read stage1={stage1_addr} uw={underwater_addr} "
          f"page=0x{page_val:02X} addr=0x{addr_val:02X} => 0x{value:04X}")
    return result

SI5345_API_VERSION = 2

- AD9253 SPI

In [376]:
def str_to_byte(value_str):
    """
    Convert a value string into one byte.

    Supports:
        '0xFF'
        'FF'
        '11111111'
        '255'
    Returns:
        b'\xFF'
    """
    value_str = value_str.strip()

    # binary
    if len(value_str) == 8 and set(value_str) <= {"0", "1"}:
        value = int(value_str, 2)
    # hex, with 0x prefix
    elif value_str.lower().startswith("0x"):
        value = int(value_str, 16)
    # hex, no 0x prefix
    elif len(value_str) == 2:
        value = int(value_str, 16)
    # decimal
    else:
        value = int(value_str)

    if not (0 <= value <= 0xFF):
        raise ValueError("Value must be between 0 and 255.")

    return bytes([value])


def config_ad9253(adc_num, test_mode, stage1_addr, underwater_addr,
                  udp_ip='192.168.31.16', udp_port=4660):
    """
    Configure one AD9253 ADC via the 0xF2 bulk-write slow-control command.

    Each byte is sent as a separate UDP write packet (matching the 1-byte
    FIFO word push in stage2 rbcp_slow_control).

    AD9253 SPI words are 3 bytes: [reg_hi, reg_lo, data].

    Parameters
    ----------
    adc_num : int
        ADC index 0~7 on the underwater board.
    test_mode : str
        "none", "1bit high", "checkerboard", "full scale" or
        "mixed frequency".
    stage1_addr : int
        Stage1 board address, 1~8.
    underwater_addr : int
        Underwater board address, 1~8.
    udp_ip : str
        SiTCP target IP.
    udp_port : int
        SiTCP target UDP port.
    """
    # 14bit, 0x56=125M, 0x50=20M
    sample_rate_reg = b'\x01\x00\x56'
    # write 0x01 into 0xFF after touching 0x100-region registers
    confirm_reg = b'\x00\xFF\x01'

    # test mode register 0x0D:
    #   0x0B: 1bit high, 0x04: checkerboard, 0x02: full scale,
    #   0x0C: mixed frequency
    if test_mode == "none":
        test_mode_reg = b'\x00\x0D\x00'
    elif test_mode == "1bit high":
        test_mode_reg = b'\x00\x0D\x0B'
    elif test_mode == "checkerboard":
        test_mode_reg = b'\x00\x0D\x04'
    elif test_mode == "full scale":
        test_mode_reg = b'\x00\x0D\x02'
    elif test_mode == "mixed frequency":
        test_mode_reg = b'\x00\x0D\x0C'
    else:
        raise ValueError(f"Invalid test_mode: {test_mode}")

    # output mode, 0x00 = offset binary
    output_mode_reg = b'\x00\x14\x00'

    config_data = (sample_rate_reg + confirm_reg + test_mode_reg +
                   output_mode_reg)
    length = len(config_data)

    def _send_byte(b):
        udp_write_byte(
            b, stage1_addr, underwater_addr,
            udp_ip=udp_ip, udp_port=udp_port
        )

    # 0xF2 command byte
    _send_byte(0xF2)

    # ADC number
    _send_byte(adc_num)

    # 2-byte big-endian length
    _send_byte((length >> 8) & 0xFF)
    _send_byte(length & 0xFF)

    # config data bytes
    for b in config_data:
        _send_byte(b)

    time.sleep(0.02)
    print(f"AD9253 config sent: {length} bytes to stage1={stage1_addr},"
          f" underwater={underwater_addr}, adc={adc_num}, "
          f"mode={test_mode}")


def usr_test_ad9253(adc_num, D1_M, D1_L, D2_M, D2_L, stage1_addr,
                    underwater_addr, udp_ip='192.168.31.16',
                    udp_port=4660):
    """
    Write AD9253 channel gain/offset registers via the 0xF2 command.

    Configures D1/D2 midscale and lowscale offsets plus test mode,
    used for ADC gain calibration.

    Parameters
    ----------
    adc_num : int
        ADC index 0~7 on the underwater board.
    D1_M, D1_L, D2_M, D2_L : str
        Midscale / lowscale offset values for channel 1 and 2.
    stage1_addr : int
        Stage1 board address, 1~8.
    underwater_addr : int
        Underwater board address, 1~8.
    udp_ip : str
        SiTCP target IP.
    udp_port : int
        SiTCP target UDP port.
    """
    sample_rate_reg = b'\x01\x00\x56'
    confirm_reg = b'\x00\xFF\x01'

    D1_M_reg = b'\x00\x1A' + str_to_byte(D1_M)
    D1_L_reg = b'\x00\x19' + str_to_byte(D1_L)
    D2_M_reg = b'\x00\x1C' + str_to_byte(D2_M)
    D2_L_reg = b'\x00\x1B' + str_to_byte(D2_L)
    test_mode_reg = b'\x00\x0D\x08'

    # output mode, 0x00 = offset binary
    output_mode_reg = b'\x00\x14\x00'

    config_data = (sample_rate_reg + confirm_reg + D1_M_reg + D1_L_reg +
                   D2_M_reg + D2_L_reg + test_mode_reg + output_mode_reg)
    length = len(config_data)

    def _send_byte(b):
        udp_write_byte(
            b, stage1_addr, underwater_addr,
            udp_ip=udp_ip, udp_port=udp_port
        )

    # 0xF2 command byte
    _send_byte(0xF2)

    # ADC number
    _send_byte(adc_num)

    # 2-byte big-endian length
    _send_byte((length >> 8) & 0xFF)
    _send_byte(length & 0xFF)

    # config data bytes
    for b in config_data:
        _send_byte(b)

    time.sleep(0.02)
    print(f"AD9253 user-test config sent: {length} bytes to "
          f"stage1={stage1_addr}, underwater={underwater_addr}, "
          f"adc={adc_num}")


def read_ad9253(adc_num, addr1, addr2, stage1_addr, underwater_addr,
                udp_ip='192.168.31.16', udp_port=4660):
    """
    Read one AD9253 register via the 0xF3 slow-control command.

    Sends the 0xF3 command, ADC number and the read address (with bit7 of
    addr1 set), then issues exactly 2 collect reads (0xC0) to retrieve one
    16-bit result from stage2 fifo_sc_tx.

    Parameters
    ----------
    adc_num : int
        ADC index 0~7 on the underwater board.
    addr1, addr2 : str
        High / low byte of the AD9253 register address, e.g. '0x00', '0x0D'.
    stage1_addr : int
        Stage1 board address, 1~8.
    underwater_addr : int
        Underwater board address, 1~8.
    udp_ip : str
        SiTCP target IP.
    udp_port : int
        SiTCP target UDP port.

    Returns
    -------
    result : str
        Hex string of the register value (2 chars, e.g. '0B').
    """
    def _send_byte(b):
        udp_write_byte(
            b, stage1_addr, underwater_addr,
            udp_ip=udp_ip, udp_port=udp_port
        )

    # 0xF3 command byte
    _send_byte(0xF3)

    # ADC number
    _send_byte(adc_num)

    # read address (bit7 of first byte = read flag)
    _send_byte(int(addr1, 16) | 0x80)
    _send_byte(int(addr2, 16))

    value = udp_read_word(
        stage1_addr, underwater_addr,
        udp_ip=udp_ip, udp_port=udp_port
    )
    result = f"{value & 0xFF:02X}"
    print(f"AD9253 read stage1={stage1_addr} uw={underwater_addr} "
          f"adc={adc_num} addr={addr1}|{addr2} => 0x{result}")
    return result

- AD9253 Bitslip

In [377]:
def ad9253_bitslip(adc_num, channel, stage1_addr, underwater_addr,
                       udp_ip='192.168.31.16', udp_port=4660,
                       time_err=None, retry_delay=0.02,
                       max_attempts=16, verbose=True):
    """
    Align one AD9253 channel output via the 0xF4 bitslip slow-control
    command.

    In UDP mode each byte is sent as a separate 1-byte write packet
    (the same 1-byte FIFO word push a UDP packet does, same mechanism
    as every other slow-control byte). After the FPGA performs one bit
    slip it pushes the captured channel pattern as a 16-bit word
    {8'b0, value} into fifo_sc_tx, which is popped with 2 collect reads
    (0xC0); the low byte is compared against the target pattern.

    The loop keeps issuing bitslips until the captured pattern matches
    the channel's target, or the timeout expires.

    Parameters
    ----------
    adc_num : int
        ADC index 0~7 on the underwater board.
    channel : str
        Channel name: 'cha0','cha1','chb0','chb1','chc0','chc1',
        'chd0','chd1'.
    stage1_addr : int
        Stage1 board address, 1~8.
    underwater_addr : int
        Underwater board address, 1~8.
    udp_ip : str
        SiTCP target IP.
    udp_port : int
        SiTCP target UDP port.
    time_err : float
        Timeout for the whole bitslip loop in seconds.

    Returns
    -------
    bool
        True if the channel aligned to the target pattern, False on
        timeout.
    """

    # 通道 → 指令映射
    channel_map = {
        "cha0": 0,
        "cha1": 1,
        "chb0": 2,
        "chb1": 3,
        "chc0": 4,
        "chc1": 5,
        "chd0": 6,
        "chd1": 7,
    }

    # 通道 → 正确数据模式映射
    target_map = {
        "cha0": "10011100",
        "cha1": "10100001",
        "chb0": "10011100",
        "chb1": "10100001",
        "chc0": "10011100",
        "chc1": "10100001",
        "chd0": "10011100",
        "chd1": "10100001",
    }
    
    if not isinstance(adc_num, int) or not (0 <= adc_num <= 7):
        raise ValueError("adc_num must be an integer between 0 and 7")

    if channel not in channel_map:
        raise ValueError(f"Invalid channel: {channel}")

    def _send_byte(b):
        udp_write_byte(
            b, stage1_addr, underwater_addr,
            udp_ip=udp_ip, udp_port=udp_port
        )

    def _read_word():
        return udp_read_word(
            stage1_addr, underwater_addr,
            udp_ip=udp_ip, udp_port=udp_port
        )

    channel_num = adc_num * 8 + channel_map[channel]

    start_time = time.time()

    for _attempt in range(max_attempts):
        if time_err is not None and time.time() - start_time > time_err:
            break

        # 0xF4 bitslip command
        _send_byte(0xF4)
        _send_byte(channel_num)

        time.sleep(0.01)

        data = _read_word()
        result = format(data & 0xFF, '08b')

        if result == target_map[channel]:
            if verbose:
                print(f"ADC{adc_num} {channel} bitslip done.")
            return True

        time.sleep(retry_delay)

    if verbose:
        print(f"ADC{adc_num} {channel} bitslip error after {max_attempts} attempts.")
    return False

- AD9253 IDELAY


In [378]:
def idelay(adc_num, channel, delay_tap,
                       stage1_addr, underwater_addr,
                       udp_ip='192.168.31.16', udp_port=4660):
    """
    Set one AD9253 channel's input IDELAY tap via the 0xF5 slow-control
    command (single-tap write, no readback).

    In UDP mode each command byte is sent as a separate 1-byte write
    packet (matching the 1-byte FIFO word push in the slow-control
    manager, same as every other slow-control byte). This writes the
    three bytes 0xF5 / channel_num / delay_tap and reads nothing back.

    Parameters
    ----------
    adc_num : int
        ADC index 0~7 on the underwater board.
    channel : str
        Channel name, one of 'cha0','cha1','chb0','chb1','chc0','chc1',
        'chd0','chd1'.
    delay_tap : int
        IDELAY tap value, 0~31.
    stage1_addr : int
        Stage1 board address, 1~8.
    underwater_addr : int
        Underwater board address, 1~8.
    udp_ip : str
        SiTCP target IP.
    udp_port : int
        SiTCP target UDP port.
    """

    channel_map = {
        "cha0": 0, "cha1": 1,
        "chb0": 2, "chb1": 3,
        "chc0": 4, "chc1": 5,
        "chd0": 6, "chd1": 7,
    }

    if not isinstance(adc_num, int) or not (0 <= adc_num <= 7):
        raise ValueError("adc_num must be an integer between 0 and 7")

    if channel not in channel_map:
        raise ValueError(f"Invalid channel: {channel}")

    if not isinstance(delay_tap, int) or not (0 <= delay_tap <= 31):
        raise ValueError("delay_tap must be an integer between 0 and 31")

    channel_num = adc_num * 8 + channel_map[channel]

    def _send_byte(b):
        udp_write_byte(
            b, stage1_addr, underwater_addr,
            udp_ip=udp_ip, udp_port=udp_port
        )

    _send_byte(0xF5)
    _send_byte(channel_num)
    _send_byte(delay_tap)


def find_best_delays(delays_matrix):
    """
    Pick the best IDELAY tap per channel from a 32-tap eye/valid matrix.

    Handles the circular sweep: appends a copy of each row so a valid run
    that wraps 31→0 is treated as one contiguous segment. The chosen tap
    is the midpoint of the longest valid run, mapped back to 0~31.

    Parameters
    ----------
    delays_matrix : numpy.ndarray
        Shape (n_channels, 32) of 0/1 validity flags per tap.

    Returns
    -------
    list
        Best tap (int 0~31) per channel, or None for a channel with no
        valid tap.
    """
    import numpy as np

    delays = []

    for ch in range(delays_matrix.shape[0]):
        row = delays_matrix[ch]
        n = len(row)

        # 拼接一份，处理环形
        extended = np.concatenate([row, row])

        max_len = 0
        max_start = 0

        curr_len = 0
        curr_start = 0

        for i in range(len(extended)):
            if extended[i] == 1:
                if curr_len == 0:
                    curr_start = i
                curr_len += 1

                if curr_len > max_len:
                    max_len = curr_len
                    max_start = curr_start
            else:
                curr_len = 0

        if max_len == 0:
            # 没有可用点
            delays.append(None)
            continue

        # 中点（取整数）
        center = max_start + max_len // 2

        # 映射回 0~31
        delay = center % n

        delays.append(delay)

    return delays


def ad9253_delay_cali(adc_num, stage1_addr, underwater_addr,
                      udp_ip='192.168.31.16', udp_port=4660,
                      time_err=None, settle_delay=0.002,
                      apply_best=True, sweep_max_attempts=12,
                      final_max_attempts=24, verify_rounds=2,
                      verbose=False):
    """
    Full 8-channel x 32-tap IDELAY calibration for one AD9253 ADC.

    For every (channel, delay_tap) it writes the tap via the 0xF5
    command and runs ad9253_bitslip (0xF4) to check whether the
    channel aligns under that tap, filling a (1,32) validity row per
    channel. The 8x32 matrix is printed are_done and passed to
    find_best_delays for the per-channel best tap.

    Note: only the sweep and best-delay selection are performed here;
    re-applying the best taps (0xF5 + ad9253_bitslip) is left as
    the commented-out loop at the end so you can inspect delays_matrix
    first.

    Parameters
    ----------
    adc_num : int
        ADC index 0~7.
    stage1_addr : int
        Stage1 board address, 1~8.
    underwater_addr : int
        Underwater board address, 1~8.
    udp_ip : str
        SiTCP target IP.
    udp_port : int
        SiTCP target UDP port.
    time_err : float
        Per-bitslip timeout passed to ad9253_bitslip.
    """

    import numpy as np

    delays_matrix = np.zeros((8, 32), dtype=int)
    channel_map = ["cha0", "cha1", "chb0", "chb1",
                   "chc0", "chc1", "chd0", "chd1"]

    for channel in range(8):
        for delay_tap in range(32):
            idelay(adc_num, channel_map[channel], delay_tap,
                              stage1_addr, underwater_addr,
                              udp_ip=udp_ip, udp_port=udp_port)
            time.sleep(settle_delay)
            if ad9253_bitslip(adc_num, channel_map[channel],
                                  stage1_addr, underwater_addr,
                                  udp_ip=udp_ip, udp_port=udp_port,
                                  time_err=time_err,
                                  retry_delay=0.01,
                                  max_attempts=sweep_max_attempts,
                                  verbose=verbose):
                delays_matrix[channel, delay_tap] = 1
            time.sleep(settle_delay)

    best_delays = find_best_delays(delays_matrix)
    aligned = []

    if apply_best:
        for channel, delay_tap in enumerate(best_delays):
            if delay_tap is None:
                aligned.append(False)
                continue
            idelay(adc_num, channel_map[channel], delay_tap,
                   stage1_addr, underwater_addr,
                   udp_ip=udp_ip, udp_port=udp_port)
            time.sleep(settle_delay)
            ok = True
            for _ in range(verify_rounds):
                ok = ad9253_bitslip(
                    adc_num, channel_map[channel],
                    stage1_addr, underwater_addr,
                    udp_ip=udp_ip, udp_port=udp_port,
                    time_err=time_err,
                    retry_delay=0.02,
                    max_attempts=final_max_attempts,
                    verbose=verbose
                )
                if not ok:
                    break
            aligned.append(ok)

    if verbose:
        print(f"ADC{adc_num} delay matrix:\n{delays_matrix}")
    print(f"ADC{adc_num} best delays: {best_delays}")
    if apply_best:
        print(f"ADC{adc_num} final bitslip: {aligned}")

    return delays_matrix, best_delays, aligned


def ad9253_delay_cali_1ch(adc_num, channel, stage1_addr,
                              underwater_addr,
                              udp_ip='192.168.31.16', udp_port=4660,
                              time_err=1.0):
    """
    Single-channel 32-tap IDELAY calibration for one AD9253 channel.

    Prints the (1,32) validity row for the given channel; the tap that
    aligns (matched by ad9253_bitslip) is marked 1. Use
    find_best_delays on this row to pick the window midpoint tap.

    Parameters
    ----------
    adc_num : int
        ADC index 0~7.
    channel : str
        Channel name, e.g. 'cha0'.
    stage1_addr : int
        Stage1 board address, 1~8.
    underwater_addr : int
        Underwater board address, 1~8.
    udp_ip : str
        SiTCP target IP.
    udp_port : int
        SiTCP target UDP port.
    time_err : float
        Per-bitslip timeout passed to ad9253_bitslip.
    """

    import numpy as np

    delays_matrix = np.zeros((1, 32), dtype=int)

    for delay_tap in range(32):
        idelay(adc_num, channel, delay_tap,
                          stage1_addr, underwater_addr,
                          udp_ip=udp_ip, udp_port=udp_port)
        time.sleep(0.05)
        if ad9253_bitslip(adc_num, channel,
                              stage1_addr, underwater_addr,
                              udp_ip=udp_ip, udp_port=udp_port,
                              time_err=time_err):
            delays_matrix[0, delay_tap] = 1
        time.sleep(0.05)

    print(delays_matrix)



In [122]:
# Configure all 8 ADCs for mixed-frequency test mode, then calibrate
# all 8 channels of each ADC and apply the selected IDELAY tap.
STAGE1_ADDR = 1
UNDERWATER_ADDR = 1
ADC_LIST = list(range(8))
ADC_CALI_VERBOSE = False
ADC_CALI_APPLY_BEST = True
ADC_SWEEP_ATTEMPTS = 12
ADC_FINAL_ATTEMPTS = 24
ADC_VERIFY_ROUNDS = 2

channel_names = [
    "cha0", "cha1", "chb0", "chb1",
    "chc0", "chc1", "chd0", "chd1",
]

adc_calibration_results = {}

for adc_num in ADC_LIST:
    print(f"\n=== ADC{adc_num}: mixed frequency ===")
    config_ad9253(
        adc_num, "mixed frequency",
        STAGE1_ADDR, UNDERWATER_ADDR
    )

    matrix, best_delays, aligned = ad9253_delay_cali(
        adc_num, STAGE1_ADDR, UNDERWATER_ADDR,
        apply_best=ADC_CALI_APPLY_BEST,
        sweep_max_attempts=ADC_SWEEP_ATTEMPTS,
        final_max_attempts=ADC_FINAL_ATTEMPTS,
        verify_rounds=ADC_VERIFY_ROUNDS,
        verbose=ADC_CALI_VERBOSE
    )

    adc_calibration_results[adc_num] = {
        "matrix": matrix,
        "best_delays": best_delays,
        "aligned": aligned,
    }

    print("delay matrix: 1=valid, 0=invalid")
    print("tap   " + " ".join(f"{tap:2d}" for tap in range(32)))
    for channel, row in zip(channel_names, matrix):
        values = " ".join(f"{int(value):2d}" for value in row)
        print(f"{channel:5s} {values}")

    print("best delays:", dict(zip(channel_names, best_delays)))
    print("aligned:", dict(zip(channel_names, aligned)))

print("\n=== Calibration summary ===")
for adc_num, result in adc_calibration_results.items():
    failed = [
        channel for channel, ok in zip(channel_names, result["aligned"])
        if not ok
    ]
    print(
        f"ADC{adc_num}: delays={result['best_delays']}, "
        f"failed={failed if failed else 'none'}"
    )


=== ADC0: mixed frequency ===
AD9253 config sent: 12 bytes to stage1=1, underwater=1, adc=0, mode=mixed frequency
ADC0 best delays: [22, 22, 22, 22, 10, 22, 10, 21]
ADC0 final bitslip: [True, True, True, True, True, True, True, True]
delay matrix: 1=valid, 0=invalid
tap    0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21 22 23 24 25 26 27 28 29 30 31
cha0   1  1  0  0  0  1  1  1  1  1  1  1  1  1  1  0  0  1  1  1  1  1  1  1  1  1  1  1  0  0  1  1
cha1   1  1  1  0  0  1  1  1  1  1  1  1  1  1  1  0  0  1  1  1  1  1  1  1  1  1  1  1  0  0  1  1
chb0   1  1  0  0  0  1  1  1  1  1  1  1  1  1  1  0  0  1  1  1  1  1  1  1  1  1  1  1  0  0  1  1
chb1   1  1  1  0  0  1  1  1  1  1  1  1  1  1  0  0  0  1  1  1  1  1  1  1  1  1  1  1  0  0  0  1
chc0   1  1  1  0  1  1  1  1  1  1  1  1  1  1  1  1  0  1  1  1  1  1  1  1  1  1  1  1  1  0  0  1
chc1   1  1  1  1  0  1  1  1  1  1  1  1  1  1  1  0  0  1  1  1  1  1  1  1  1  1  1  1  0  0  0  1
chd0   1  1  1  0 

- DAC128S085


In [91]:
def config_dac128s085(dac_num, channel, voltage,
                        stage1_addr, underwater_addr,
                        udp_ip='192.168.31.16', udp_port=4660):
    """
    配置 DAC128S085。

    在 UDP 模式下每个命令字节被当作独立的 1 字节写包发送（与其它慢控
    字节的 1 字节 FIFO 压栈机制一致）。写入 0xF6 / dac_num / WTM 模式
    字节，再写入 0xF6 / dac_num / 16 位大端数据。无读回。

    Parameters
    ----------
    dac_num : int
        DAC 芯片号，0~7。
    channel : str
        通道名，'A'/'B'/'C'/'D'/'ALL'。
    voltage : float
        输出电压，0~3.3 V。
    stage1_addr : int
        Stage1 板地址，1~8。
    underwater_addr : int
        水下板地址，1~8。
    udp_ip : str
        SiTCP 目标 IP。
    udp_port : int
        SiTCP 目标 UDP 端口。
    """

    if not isinstance(dac_num, int) or not (0 <= dac_num <= 7):
        raise ValueError("dac_num must be an integer between 0 and 7")

    if voltage < 0 or voltage > 3.3:
        raise ValueError("电压值必须在 0~3.3 V 之间")

    # 12 位 DAC 码
    D = int(voltage * 4096 / 3.3)

    # CMD 决定通道
    if channel.upper() == "A":
        cmd = 0b0011
    elif channel.upper() == "B":
        cmd = 0b0010
    elif channel.upper() == "C":
        cmd = 0b0001
    elif channel.upper() == "D":
        cmd = 0b0000
    elif channel.upper() == "ALL":
        cmd = 0b1100
    else:
        raise ValueError("不支持的通道，有效通道为: A, B, C, D, ALL")

    # DB[15:12] = cmd, DB[11:0] = D
    db = (cmd << 12) | (D & 0xFFF)

    def _send_byte(b):
        udp_write_byte(
            b, stage1_addr, underwater_addr,
            udp_ip=udp_ip, udp_port=udp_port
        )

    # 0xF6 命令
    _send_byte(0xF6)
    _send_byte(dac_num)

    # WTM 模式配置
    _send_byte(0b10010000)
    _send_byte(0b00000000)

    # 写入 16 位数据（大端）
    _send_byte(0xF6)
    _send_byte(dac_num)
    _send_byte((db >> 8) & 0xFF)
    _send_byte(db & 0xFF)


def trig_random_2ch(comp_input_static,
                     stage1_addr, underwater_addr,
                     udp_ip='192.168.31.16', udp_port=4660):
    """
    随机选中两个通道并写入比较器静态输入电压。

    在 0~31 中随机取两个不重复整数，映射为 (dac_num, channel)，
    再调用 config_dac128s085 写入 comp_input_static 对应值
    （加 0.015 V 偏移）。先对所有 8 块 DAC 的 ALL 通道写 2.1 V。

    Parameters
    ----------
    comp_input_static : list[list[float]]
        8×4 的比较器静态输入电压矩阵。
    stage1_addr : int
        Stage1 板地址，1~8。
    underwater_addr : int
        水下板地址，1~8。
    udp_ip : str
        SiTCP 目标 IP。
    udp_port : int
        SiTCP 目标 UDP 端口。
    """

    import random

    # 随机生成 0~31
    num1, num2 = random.sample(range(32), 2)

    # 计算 dac_num 和 channel
    dac_num1 = num1 // 4
    channel1 = num1 % 4
    dac_num2 = num2 // 4
    channel2 = num2 % 4

    print(f"随机数1: {num1}")
    print(f"dac_num1: {dac_num1}")
    print(f"channel1: {channel1}")
    print(f"随机数2: {num2}")
    print(f"dac_num2: {dac_num2}")
    print(f"channel2: {channel2}")

    # 先将所有 DAC 的 ALL 通道配置为 2.1 V
    for i in range(8):
        config_dac128s085(i, "ALL", 2.1,
                            stage1_addr, underwater_addr,
                            udp_ip=udp_ip, udp_port=udp_port)

    config_dac128s085(dac_num1, channel1,
                       comp_input_static[dac_num1][channel1] + 0.015,
                       stage1_addr, underwater_addr,
                       udp_ip=udp_ip, udp_port=udp_port)
    config_dac128s085(dac_num2, channel2,
                       comp_input_static[dac_num2][channel2] + 0.015,
                       stage1_addr, underwater_addr,
                       udp_ip=udp_ip, udp_port=udp_port)



- 时钟同步

# Test

### Si5345

In [ ]:
si5345_config_data_125M = parse_si5345_txt_file("Si5345-RevD-5345CONF-Registers_125M.txt")
length_bytes = len(si5345_config_data_125M).to_bytes(2, byteorder='big')

In [382]:
si5345_config_data_125M = parse_si5345_txt_file("Si5345-RevD-5345CONF-Registers_125M.txt")
config_si5345(si5345_config_data_125M, 1, 1)

Si5345 config sent: 2176 bytes to stage1=1, underwater=1


2176

In [402]:
if (globals().get('DAQ_RBCP_API_VERSION') != 3 or
        globals().get('SI5345_API_VERSION') != 2):
    raise RuntimeError(
        'Old notebook functions are loaded. Restart Kernel and Run All.'
    )

print("Si5345 0x0051 =", read_si5345(0x01, 0x1C, 1, 1))

Si5345 read stage1=1 uw=1 page=0x01 addr=0x1C => 0x0006
Si5345 0x0051 = 06


### DAC12S085

In [128]:
comp_input_static = np.array([[1.88, 1.88, 1.88, 1.89], [1.824, 1.823, 1.817, 1.823], [1.87, 1.88, 1.88, 1.88], [1.83, 1.83, 1.83, 1.83], 
                              [1.86, 1.87, 1.87, 1.86], [1.83, 1.83, 1.83, 1.83], [1.85, 1.85, 1.85, 1.85], [1.84, 1.84, 1.84, 1.83]])
dac_value = comp_input_static - 0.04

In [129]:
channel_map = {
    "a": 0,
    "b": 1,
    "c": 2,
    "d": 3,
}
for i in range(8):
    for j in range(4):
        config_dac128s085(i, list(channel_map.keys())[j], dac_value[i][j], 1, 1)

In [131]:
# Configure all DAC128S085 channels using the reference values from
# Veto_32ch_tdc/uart_32ch_new.ipynb.
comp_input_static_uart = np.array([
    [1.88,  1.88,  1.88,  1.89],
    [1.824, 1.823, 1.817, 1.823],
    [1.87,  1.88,  1.88,  1.88],
    [1.87,  1.84,  1.90,  1.89],
    [1.86,  1.87,  1.87,  1.86],
    [1.83,  1.83,  1.83,  1.83],
    [1.85,  1.85,  1.85,  1.85],
    [1.84,  1.84,  1.84,  1.83],
])

dac_value_uart = comp_input_static_uart + 0.05
dac_channels = ["a", "b", "c", "d"]

dac_configuration = []

for dac_num in range(8):
    for channel_index, channel in enumerate(dac_channels):
        voltage = float(dac_value_uart[dac_num][channel_index])
        config_dac128s085(
            dac_num, channel, voltage,
            1, 1
        )
        dac_configuration.append({
            "dac_num": dac_num,
            "channel": channel,
            "voltage": voltage,
        })
        print(
            f"DAC{dac_num} channel {channel.upper()}: "
            f"{voltage:.3f} V"
        )

# RBCP ACK confirms receipt; allow the final DAC SPI transaction to finish.
time.sleep(0.05)
print(f"Configured {len(dac_configuration)} DAC channels.")


DAC0 channel A: 1.930 V
DAC0 channel B: 1.930 V
DAC0 channel C: 1.930 V
DAC0 channel D: 1.940 V
DAC1 channel A: 1.874 V
DAC1 channel B: 1.873 V
DAC1 channel C: 1.867 V
DAC1 channel D: 1.873 V
DAC2 channel A: 1.920 V
DAC2 channel B: 1.930 V
DAC2 channel C: 1.930 V
DAC2 channel D: 1.930 V
DAC3 channel A: 1.920 V
DAC3 channel B: 1.890 V
DAC3 channel C: 1.950 V
DAC3 channel D: 1.940 V
DAC4 channel A: 1.910 V
DAC4 channel B: 1.920 V
DAC4 channel C: 1.920 V
DAC4 channel D: 1.910 V
DAC5 channel A: 1.880 V
DAC5 channel B: 1.880 V
DAC5 channel C: 1.880 V
DAC5 channel D: 1.880 V
DAC6 channel A: 1.900 V
DAC6 channel B: 1.900 V
DAC6 channel C: 1.900 V
DAC6 channel D: 1.900 V
DAC7 channel A: 1.890 V
DAC7 channel B: 1.890 V
DAC7 channel C: 1.890 V
DAC7 channel D: 1.880 V
Configured 32 DAC channels.


### Config ADC

In [370]:
for i in range(8):
    config_ad9253(i, "mixed frequency", 1, 1)
    read_ad9253(i, "00", "0D", 1, 1)

AD9253 config sent: 12 bytes to stage1=1, underwater=1, adc=0, mode=mixed frequency
AD9253 read stage1=1 uw=1 adc=0 addr=00|0D => 0x0C
AD9253 config sent: 12 bytes to stage1=1, underwater=1, adc=1, mode=mixed frequency
AD9253 read stage1=1 uw=1 adc=1 addr=00|0D => 0x0C
AD9253 config sent: 12 bytes to stage1=1, underwater=1, adc=2, mode=mixed frequency
AD9253 read stage1=1 uw=1 adc=2 addr=00|0D => 0x0C
AD9253 config sent: 12 bytes to stage1=1, underwater=1, adc=3, mode=mixed frequency
AD9253 read stage1=1 uw=1 adc=3 addr=00|0D => 0x0C
AD9253 config sent: 12 bytes to stage1=1, underwater=1, adc=4, mode=mixed frequency
AD9253 read stage1=1 uw=1 adc=4 addr=00|0D => 0x0C
AD9253 config sent: 12 bytes to stage1=1, underwater=1, adc=5, mode=mixed frequency
AD9253 read stage1=1 uw=1 adc=5 addr=00|0D => 0x0C
AD9253 config sent: 12 bytes to stage1=1, underwater=1, adc=6, mode=mixed frequency
AD9253 read stage1=1 uw=1 adc=6 addr=00|0D => 0x0C
AD9253 config sent: 12 bytes to stage1=1, underwater=1,

In [ ]:
adc0_delay_value = [22, 21, 10, 22, 10, 10, 10, 20]
adc1_delay_value = [11, 9, 24, 10, 10, 10, 17, 8]
adc2_delay_value = [5, 5, 5, 18, 5, 18, 5, 7]
adc3_delay_value = [19, 8, 20, 21, 19, 9, 19, 23]
adc4_delay_value = [13, 25, 13, 25, 13, 24, 13, 25]
adc5_delay_value = [13, 12, 12, 12, 25, 12, 12, 12]
adc6_delay_value = [24, 23, 24, 21, 10, 24, 10, 12]
adc7_delay_value = [8, 9, 8, 8, 8, 8, 0, 7]
adc_delay_value = [adc0_delay_value, adc1_delay_value, adc2_delay_value, adc3_delay_value, adc4_delay_value, adc5_delay_value, adc6_delay_value, adc7_delay_value]

In [188]:
# Apply the predefined delay taps and run bitslip for all ADC channels.
STAGE1_ADDR = 1
UNDERWATER_ADDR = 1
BITSLIP_ATTEMPTS = 24
BITSLIP_ROUNDS = 2

channel_names = [
    "cha0", "cha1", "chb0", "chb1",
    "chc0", "chc1", "chd0", "chd1",
]

if len(adc_delay_value) != 8 or any(len(values) != 8 for values in adc_delay_value):
    raise ValueError("adc_delay_value must be an 8 x 8 matrix")

adc_delay_apply_results = {}

for adc_num, delays in enumerate(adc_delay_value):
    print(f"\n=== ADC{adc_num} ===")
    aligned = []

    for channel, delay_tap in zip(channel_names, delays):
        idelay(
            adc_num, channel, delay_tap,
            STAGE1_ADDR, UNDERWATER_ADDR
        )
        time.sleep(0.01)

        ok = True
        for _round in range(BITSLIP_ROUNDS):
            ok = ad9253_bitslip(
                adc_num, channel,
                STAGE1_ADDR, UNDERWATER_ADDR,
                retry_delay=0.02,
                max_attempts=BITSLIP_ATTEMPTS,
                verbose=False
            )
            if not ok:
                break

        aligned.append(ok)
        print(
            f"ADC{adc_num} {channel}: delay={delay_tap:2d}, "
            f"aligned={ok}"
        )

    adc_delay_apply_results[adc_num] = {
        "delays": list(delays),
        "aligned": aligned,
    }

print("\n=== Delay apply summary ===")
for adc_num, result in adc_delay_apply_results.items():
    failed = [
        channel for channel, ok in zip(channel_names, result["aligned"])
        if not ok
    ]
    print(
        f"ADC{adc_num}: delays={result['delays']}, "
        f"failed={failed if failed else 'none'}"
    )


=== ADC0 ===
ADC0 cha0: delay=22, aligned=False
ADC0 cha1: delay=21, aligned=False
ADC0 chb0: delay=10, aligned=False
ADC0 chb1: delay=22, aligned=False
ADC0 chc0: delay=10, aligned=False
ADC0 chc1: delay=10, aligned=False
ADC0 chd0: delay=10, aligned=False
ADC0 chd1: delay=20, aligned=False

=== ADC1 ===
ADC1 cha0: delay=11, aligned=False
ADC1 cha1: delay= 9, aligned=False
ADC1 chb0: delay=24, aligned=False
ADC1 chb1: delay=10, aligned=False
ADC1 chc0: delay=10, aligned=False
ADC1 chc1: delay=10, aligned=False
ADC1 chd0: delay=17, aligned=False
ADC1 chd1: delay= 8, aligned=False

=== ADC2 ===
ADC2 cha0: delay= 5, aligned=False
ADC2 cha1: delay= 5, aligned=False
ADC2 chb0: delay= 5, aligned=False
ADC2 chb1: delay=18, aligned=False
ADC2 chc0: delay= 5, aligned=False
ADC2 chc1: delay=18, aligned=False
ADC2 chd0: delay= 5, aligned=False
ADC2 chd1: delay= 7, aligned=False

=== ADC3 ===
ADC3 cha0: delay=19, aligned=False
ADC3 cha1: delay= 8, aligned=False
ADC3 chb0: delay=20, aligned=False

KeyboardInterrupt: 

In [13]:
client = socket.socket(family=socket.AF_INET, type=socket.SOCK_STREAM)
client.settimeout(5)
client.connect(('192.168.31.16', 24))
client

<socket.socket fd=1432, family=2, type=1, proto=0, laddr=('192.168.31.12', 6157), raddr=('192.168.31.16', 24)>